<h1>Importing all the libraries</h1>
<p>And getting library versions, along with device connected</p>

In [1]:
import os

os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"
os.environ["HIP_VISIBLE_DEVICES"] = "0"
os.environ["ROCR_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_DISABLE_RICH"] = "1"

import transformers, datasets, peft, trl, accelerate, torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, logging)
from trl import SFTConfig, SFTTrainer
import torch
import bitsandbytes as bnb
# pip install transformers datasets peft trl accelerate bitsandbytes trl transformers datasets peft
print("bitsandbytes:", bnb.__version__)
print("transformers: "+str(transformers.__version__))
print("datasets: "+str(datasets.__version__))
print("peft: "+str(peft.__version__))
print("trl: "+str(trl.__version__))
print("accelerate: "+str(accelerate.__version__))


print("torch cuda version:", torch.version.cuda)
print("cuda arch supported:", torch.cuda.get_device_capability(0))

logging.set_verbosity_error()
print("Cuda availablitiy: "+str(torch.cuda.is_available()))
if torch.cuda.is_available():
    print("Device is: "+str(torch.cuda.get_device_name(0)))

use_gpu = (torch.version.hip is not None) or torch.cuda.is_available()
print(use_gpu)
torch.cuda.empty_cache()
print("HIP:", torch.version.hip)



print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))


from huggingface_hub import login
# login("")


Failed to load /home/puffle/miniconda3/envs/torchrocm72/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/puffle/miniconda3/envs/torchrocm72/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/puffle/miniconda3/envs/torchrocm72/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/puffle/miniconda3/envs/torchrocm72/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
W0827 23:11:33.181000 60804 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0827 23:11:33.191000 60804 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque

bitsandbytes: 0.49.2
transformers: 5.14.1
datasets: 5.0.0
peft: 0.19.1
trl: 1.9.0
accelerate: 1.14.0
torch cuda version: None
cuda arch supported: (10, 3)
Cuda availablitiy: True
Device is: AMD Radeon RX 6700 XT
True
HIP: 7.2.53211
1
AMD Radeon RX 6700 XT


<h1>Loading in model and dataset</h1>
<p>Setting parameters and formatting the dataset.</p>

In [2]:
# Choosing model and dataset and loading it.
model = "unsloth/Phi-3-mini-4k-instruct"
datasetFormatted = "milistu/robot-instructions"
trainingData = load_dataset(datasetFormatted, split="train")
print(trainingData)
print("Columns:", trainingData.column_names)
print("First example:")
print(trainingData[0])



def formatExample(example):
    return {"text": ( 
                    f"<|im_start|>user\n{example['input']}<|im_end|>\n"
                    f"<|im_start|>assistant\n{example['output']}"
                    )}


trainingData = trainingData.map(formatExample)


# Tests if its formatted fine with the text column
if "text" not in trainingData.column_names:
    raise ValueError(
        f"Expected a column named 'text', but found: {trainingData.column_names}. "
        "Update the dataset formatting step before training."
    )

print(trainingData[0]["text"][:1000])


# Lora parameters
loraHyperR = 4 #Rank
loraHyperAlpha = 8 #
loraHyperDropout = 0.05

# Bits and Bytes args
enable4Bit = False

# Training Args
resultsDir = "./results"
epochsCount = 1

trainBatchSize = 1
evalBatchSize = 1
accumulationSteps = 64
maxSteps = 50


gradNormLimit = 0.3
trainLearningRate = 2e-4
decayRate = 0.001

optimizerType = "adamw_torch"
schedulerType = "cosine"

warmupPercentage = 0.03
lengthGrouping = True
logInterval = 5
checkpointingFlag = True
enableFp16 = False
enableBf16 = True

# Supervised fine tuning args
enablePacking = False
sequenceLengthMax = 256

Dataset({
    features: ['input', 'output'],
    num_rows: 887
})
Columns: ['input', 'output']
First example:
{'input': 'Rotate joint 2 by 30 degrees, joint 7 by 45 degrees, and joint 3 by π/4 radians', 'output': '[{"function": "move_joint", "kwargs": {"joint": [2, 7], "angle": [0.5235987755982988, 0.7853981633974483]}}, {"function": "move_joint", "kwargs": {"joint": [3], "angle": [0.785398]}}]'}
<|im_start|>user
Rotate joint 2 by 30 degrees, joint 7 by 45 degrees, and joint 3 by π/4 radians<|im_end|>
<|im_start|>assistant
[{"function": "move_joint", "kwargs": {"joint": [2, 7], "angle": [0.5235987755982988, 0.7853981633974483]}}, {"function": "move_joint", "kwargs": {"joint": [3], "angle": [0.785398]}}]


<h1>Loading in the Model</h1>

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


phiTokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
if phiTokenizer.pad_token is None:
    phiTokenizer.pad_token = phiTokenizer.eos_token
phiTokenizer.padding_side = "right"

# Defining model
phiModel = AutoModelForCausalLM.from_pretrained(
    model,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map={"":0},
)

phiModel = phiModel.to(dtype=torch.bfloat16)
phiModel.config.use_cache = False
print("loaded")
phiModel.generation_config.use_cache = False
phiModel.config.pretraining_tp = 1

phiModel.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
phiModel.enable_input_require_grads()

print("Tokenizer loaded.")
print("pad_token:", phiTokenizer.pad_token)
print("eos_token:", phiTokenizer.eos_token)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded
Tokenizer loaded.
pad_token: <|placeholder6|>
eos_token: <|endoftext|>


<h1>Setting the lora config hyperparameters</h1>

In [4]:
# Creating QLoRA config
peft_setup = LoraConfig(
    lora_alpha=loraHyperAlpha,
    lora_dropout=loraHyperDropout,
    r=loraHyperR,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        # "k_proj",
        "v_proj",
        # "o_proj",
        "gate_proj",
        # "up_proj",
        # "down_proj",
    ],
)

print(peft_setup)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=4, target_modules={'gate_proj', 'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [5]:
from peft import get_peft_model

phiModelMulti = get_peft_model(phiModel, peft_setup)
phiModel = get_peft_model(phiModel, peft_setup)
phiModel.print_trainable_parameters()

trainable params: 3,014,656 || all params: 3,824,094,208 || trainable%: 0.0788


/home/puffle/miniconda3/envs/torchrocm72/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/puffle/miniconda3/envs/torchrocm72/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


<h1>Setting the training parameters</h1>

In [6]:
train_args = SFTConfig(
    output_dir=resultsDir,
    num_train_epochs=epochsCount,
    per_device_train_batch_size=trainBatchSize,
    per_device_eval_batch_size=evalBatchSize,
    gradient_accumulation_steps=accumulationSteps,
    learning_rate=trainLearningRate,
    weight_decay=decayRate,
    optim=optimizerType,
    loss_type="nll",
    save_strategy="no",
    logging_steps=logInterval,
    fp16=enableFp16,
    bf16=enableBf16,
    max_grad_norm=gradNormLimit,
    max_steps=maxSteps,
    warmup_ratio=warmupPercentage,
    lr_scheduler_type=schedulerType,
    gradient_checkpointing=checkpointingFlag,
    report_to=[],
    dataset_text_field="text",
    max_length=sequenceLengthMax,
    packing=enablePacking,
    eval_strategy="no",
)

print(train_args)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=True,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_che

<h1>Making the supervised trainer</h1>
<p>and training the model</p>

In [7]:
# restart kernel first

from trl import SFTTrainer
torch.cuda.empty_cache()

phiSfttTrainer = SFTTrainer(
    model=phiModel,
    args=train_args,
    train_dataset=trainingData,
    processing_class=phiTokenizer,
)
print("trainer ok")
variableModel = phiModel

trainer ok


In [8]:

# Checking that everything went smoothly
print("SFTTrainer created successfully.")
sample_batch = next(iter(phiSfttTrainer.get_train_dataloader()))
print(sample_batch.keys())
print("input_ids shape:", sample_batch["input_ids"].shape)
print("labels shape:", sample_batch["labels"].shape)

SFTTrainer created successfully.
dict_keys(['input_ids', 'labels', 'attention_mask'])
input_ids shape: torch.Size([1, 50])
labels shape: torch.Size([1, 50])


In [9]:
for name, param in phiSfttTrainer.model.named_parameters():
    print(name, param.device, param.dtype)
    break

base_model.model.model.embed_tokens.weight cuda:0 torch.bfloat16


In [10]:
trainable = 0
total = 0

for n, p in phiSfttTrainer.model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print(f"trainable: {trainable:,}")
print(f"total: {total:,}")
print(f"percentage: {100*trainable/total:.4f}%")

trainable: 3,014,656
total: 3,824,094,208
percentage: 0.0788%


In [11]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   7299 MiB |   7305 MiB |  14610 MiB |   7311 MiB |
|       from large pool |   7287 MiB |   7288 MiB |  14575 MiB |   7288 MiB |
|       from small pool |     11 MiB |     17 MiB |     34 MiB |     23 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   7299 MiB |   7305 MiB |  14610 MiB |   7311 MiB |
|       from large pool |   7287 MiB |   7288 MiB |  14575 MiB |

In [12]:

print("CUDA devices:", torch.cuda.device_count())

for name, param in phiSfttTrainer.model.named_parameters():
    if param.requires_grad:
        print(name, param.device, param.dtype)
        break

print("model device:")
print(next(phiSfttTrainer.model.parameters()).device)

CUDA devices: 1
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight cuda:0 torch.float32
model device:
cuda:0


Testing the trainer

In [13]:
print(phiSfttTrainer.args.device)
print(phiSfttTrainer.model.hf_device_map if hasattr(phiSfttTrainer.model, "hf_device_map") else "no map")

cuda:0
no map


In [14]:
x=torch.randn(1024,1024,device="cuda:0",dtype=torch.float32)
y=x@x

In [15]:
phiModel.train()

batch = next(iter(phiSfttTrainer.get_train_dataloader()))
batch = {k:v.to("cuda:0") for k,v in batch.items()}

optimizer = torch.optim.AdamW(
    [p for p in phiModel.parameters() if p.requires_grad],
    lr=2e-5
)

print("forward")
out = phiModel(**batch)
loss = out.loss
print("loss:", loss.item())

print("backward")
loss.backward()
print("backward passed")

print("optimizer")
optimizer.step()
print("optimizer passed")

forward
loss: 2.0608766078948975
backward
backward passed
optimizer
optimizer passed


In [16]:
print(phiSfttTrainer.args)
print(phiSfttTrainer.optimizer)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=True,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_che

Actually Training the model

In [17]:
# trainingResult = phiSfttTrainer.train()
# print(trainingResult)

torch.cuda.empty_cache()

print("before")
trainingResult = phiSfttTrainer.train()
print("after")


before
{'loss': '1.605', 'grad_norm': '0.6904', 'learning_rate': '0.0001991', 'entropy': '0.9898', 'mean_token_accuracy': '0.705', 'num_tokens': '2.665e+04', 'epoch': '0.3608'}
{'loss': '1.327', 'grad_norm': '0.7902', 'learning_rate': '0.0001897', 'entropy': '1.018', 'mean_token_accuracy': '0.7483', 'num_tokens': '5.26e+04', 'epoch': '0.7215'}
{'loss': '0.9579', 'grad_norm': '0.4885', 'learning_rate': '0.0001707', 'entropy': '0.9199', 'mean_token_accuracy': '0.8014', 'num_tokens': '7.923e+04', 'epoch': '1.072'}
{'loss': '0.7538', 'grad_norm': '0.3931', 'learning_rate': '0.0001442', 'entropy': '0.7849', 'mean_token_accuracy': '0.854', 'num_tokens': '1.064e+05', 'epoch': '1.433'}
{'loss': '0.6164', 'grad_norm': '0.2879', 'learning_rate': '0.0001131', 'entropy': '0.63', 'mean_token_accuracy': '0.8814', 'num_tokens': '1.327e+05', 'epoch': '1.794'}
{'loss': '0.5476', 'grad_norm': '0.3569', 'learning_rate': '8.049e-05', 'entropy': '0.5425', 'mean_token_accuracy': '0.8878', 'num_tokens': '1.5

In [18]:

# Instead of saving the whole model we just save the lora adapter to save space
phiSfttTrainer.model.save_pretrained("./phiRoboticsLoraAdapterSet1")
phiTokenizer.save_pretrained("./phiRoboticsLoraAdapterSet1")

print("LoRA adapter saved to ./phiRoboticsLoraAdapterSet1")

LoRA adapter saved to ./phiRoboticsLoraAdapterSet1


Chatting with the model

In [ ]:
import torch




# Generates commands for robotics on the users request 
def genRobotics(question, maxNewTokens=220):
    # prompt = formatPhiStylePrompt(question)
    prompt =        f"<|im_start|>user\n{question}<|im_end|>\n\n<|im_start|>assistant\n"
    phiSfttTrainer.model.eval()

    inputs = phiTokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=256,
    )

    device = next(phiSfttTrainer.model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output_ids = phiSfttTrainer.model.generate(
            **inputs,
            max_new_tokens=maxNewTokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=phiTokenizer.eos_token_id,
            eos_token_id=phiTokenizer.eos_token_id,
            use_cache=False,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    answer = phiTokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return answer

print(genRobotics("Get the tempurature reading of the Radioactive Core's twelfth rod"))

In [30]:
# Test yourself
print(genRobotics("Move the bicep joint 20 degrees."))

To move your bicep muscle by approximately 20 degrees, you can perform a simple elbow flexion exercise. Here's how to do it:

1. Stand or sit with good posture and relaxed shoulders. Place one hand on top of the other for stability if needed.
2. Keeping your upper arm stationary (elbow should be close to your torso), slowly lift your lower arm towards your shoulder while keeping your palm facing downward throughout the movement. This action engages your biceps brachii muscle – the primary muscle in your front arm responsible for this motion.
3. Stop when your fist is about level with your chest; ensure not overextending as that would involve moving more than just the angle at which your forearm meets your upper arm. A general guideline could range from roughly 45-60 degrees depending on individual anatomy, but aim for around halfway up since full extension isn’t being targeted here. Remember though—everyone has different
